In [ ]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [ ]:
#carico i file parquet del dataset green per l'anno 2020 e 2021 e tutti i mesi 
df_green = spark.read.parquet("data/pq/green/*/*")

In [ ]:
df_green.printSchema()

In [ ]:
#carico i file parquet del dataset yellow per l'anno 2020 e 2021 e tutti i mesi 
df_yellow = spark.read.parquet("data/pq/yellow/*/*")

In [ ]:
#si vuole eseguire una query del modulo 4, che abbiamo eseguito su dbt
# la query è quella che unisce i 2 dataset (green e yellow) 

#Per farla con Spark si utilizza set(df_green.columns) => 
# .columns: restituisce una lista con i nomi delle colonne del Dataframe
# set: crea un set con il nome delle colonne del Dataframe restituito da .columns

#NOTA: con & fa l'intersezione tra i 2 set, 
# quindi restituisce solo i nomi delle colonne che sono presenti in entrambi i Dataframe.
set(df_green.columns) & set(df_yellow.columns) 

NOTA: in questa "UNION" tra Dataframe, mancano le colonne pickup_datetime e dropoff_datetime perché nei 2 dataset hanno 1 lettera diversa. 
Quindi adesso si modificano i nomi delle colonne nei 2 Dataframe e si rifà la "UNION" tramite il comando di sopra. 

In [ ]:
df_green = df_green \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

La modifica del nome della colonna si fa con il comando: 
.withColumnRenamed("NomeVeccchioColonna", "NomeNuovo")

In [ ]:
df_yellow = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

In [ ]:
#si esegue di nuovo (per comodità l'ho messo qui sotto)
set(df_green.columns) & set(df_yellow.columns) 

NOTA: com'è possibile vedere nell'ultimo comando, l'ordine delle colonne è cambiato. 
Cioè le 2 nuove colonne che sono state inserite non sono nel corretto ordine, per come era nel df_green e df_yellow. 
Per risolvere questo problem si definisce la lista: common_columns [] dove si inseriscono tutte le colonne uguale tra i 2 dataframe: 

In [ ]:
common_columns = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns: 
        common_columns.append(col)


In [ ]:
df_green.select(common_columns).show()

Nel mod. 4: avevamo aggiunto una nuova colonna al Dataset in cui si definitiva il tipo di servizio (service_type), ovvero serve per definire se quella colonna è del Dataset dei taxi green o yellow. 
Qui si fa lo stesso in questo. 
Si aggiunge una nuova colonna tramite la funzione F.lit() 

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_green_sel = df_green.select(common_columns) \
    .withColumn('service_type', F.lit('green'))

F.lit(): La funzione lit() in Spark viene utilizzata per creare una nuova colonna con un valore costante. Fa parte del modulo *pyspark.sql.functions* ed è particolarmente utile quando è necessario aggiungere una colonna con un valore fisso a un DataFrame. 
Questa funzione viene spesso utilizzata in combinazione con altre trasformazioni, come withColumn().


In [ ]:
#stessa cosa però per il dataset yellow
df_yellow_sel = df_yellow.select(common_columns) \
    .withColumn('service_type', F.lit('yellow'))

In [ ]:
#Adesso si crea il df trips_data come UNION tra i 2 dataset (green e yellow)

df_trips_data = df_green_sel.unionAll(df_yellow_sel)

In [ ]:
#vediamo quanti record ci sono per il campo service_type per il dataset green e yellow
df_trips_data.groupBy('service_type').count().show()

Si vuole fare la stessa cosa ma tramite codice SQL utilizzando spark.sql(QUERY)
Per fare questo però bisogna creare una tabella temporanea. 
Questo si fa tramite il metodo: *.registerTempTable('nomeTabella')*

In [ ]:
df_trips_data.registerTempTable('trips_data')

In [ ]:
spark.sql("""
SELECT service_type, count(*) as total
FROM trips_data
GROUP BY service_type
""").show()

In [ ]:
#adesso che tutto è "pronto" è possibile eseguire la query del modulo 4: 

df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

In [ ]:
df_result.write.parquet("data/report/revenue/") 

Quando si creano dei file parquet (come fatto nell'ultimo comando), c'è il rischio che si creano tanti piccoli file (problema: di small partition in Spark). 
Per risolvere questo si può fare la colasce (unione dei vari file parquet in unico file).
.coalesce(n) -> n = numero di file in cui raggruppare i vari file piccoli.

In [ ]:
df_result.coalesce(1).write.parquet("data/report/revenue/", mode="overwrite")